# Reviewing the Local ChromaDB

This notebook is a **read-only** tool for inspecting what is currently stored in the project's
ChromaDB vector database — the same database populated by `01_pdf_ingestion.ipynb` and the
Streamlit app (`clients/streamlit_app.py`).

Use it to answer questions like:
- Which **collections** exist on the server?
- Which **documents (PDFs)** are stored in a collection, and under what **label**?
- What **metadata** was written with each chunk, and what do the raw records look like?
- How do I **filter** records by metadata (file, label, chunk type, page)?

None of these cells modify data — they only query it.

**Connection settings** come from `config/settings.yaml` (`chromadb.host`, `chromadb.port`,
`chromadb.collection_name`). Run this inside the dev container so the `chromadb` service is
reachable.

## Setup

In [9]:
# Auto-reload edited modules (e.g. rag/*) without restarting the kernel.
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import chromadb
from rag.config import load_config
from rag.store import ChromaStore

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [10]:
config = load_config()
print(f"ChromaDB server: {config.chromadb.host}:{config.chromadb.port}")
print(f"Default collection: {config.chromadb.collection_name}")

client = chromadb.HttpClient(
    host=config.chromadb.host,
    port=config.chromadb.port,
)
client.heartbeat()  # returns a timestamp if the server is reachable

ChromaDB server: chromadb:8000
Default collection: financial_reports


1783916521880048227

## 1. List all collections

A ChromaDB server can hold many **collections** (independent groups of vectors). The notebooks
use `financial_reports`; the Streamlit app creates ephemeral `streamlit_<id>` collections in its
ad-hoc mode.

> **Version note:** `client.list_collections()` returns name *strings* in some ChromaDB releases
> and `Collection` *objects* in others (which aren't sortable). The helper below normalizes to
> names so it works either way.

In [11]:
def collection_names(client) -> list[str]:
    """Sorted collection names, robust across ChromaDB versions."""
    names = [
        c if isinstance(c, str) else c.name
        for c in client.list_collections()
    ]
    return sorted(names)


names = collection_names(client)
print(f"{len(names)} collection(s) on the server:\n")
for name in names:
    col = client.get_collection(name)
    print(f"  - {name}: {col.count()} chunks")

2 collection(s) on the server:

  - financial_reports: 403 chunks
  - streamlit_e615510979ac: 0 chunks


## 2. Inspect a collection — which documents (PDFs) are stored?

Each chunk carries a `source_file` (the file it came from) and a human-friendly `label` in its
metadata. Aggregating chunks by `source_file` shows the distinct documents in a collection.

In [12]:
# Choose which collection to inspect (defaults to the config's collection).
collection_name = config.chromadb.collection_name
collection = client.get_collection(collection_name)
total = collection.count()
print(f"Collection '{collection_name}': {total} chunks\n")

# Aggregate chunks by source_file, capturing the label of each document.
docs: dict[str, dict] = {}
for offset in range(0, total, 100):
    batch = collection.get(include=["metadatas"], limit=100, offset=offset)
    for meta in batch["metadatas"]:
        meta = meta or {}
        source = meta.get("source_file", "unknown")
        if source not in docs:
            docs[source] = {"label": meta.get("label", ""), "chunks": 0}
        docs[source]["chunks"] += 1

print(f"{len(docs)} document(s):")
for source, info in docs.items():
    label = info["label"] or "\u2014"
    print(f"  - {source}  [label: {label}]  ({info['chunks']} chunks)")

Collection 'financial_reports': 403 chunks

1 document(s):
  - form-10-q.pdf  [label: Disney Q2 2026 Earnings]  (403 chunks)


## 3. Peek at raw records

Look at a few individual records to see exactly what is stored: the chunk **id**, its **metadata**,
and a **text excerpt**. `collection.get()` retrieves stored data directly (no embedding/query
needed).

In [13]:
sample = collection.get(include=["documents", "metadatas"], limit=3)
for i, (rid, doc, meta) in enumerate(
    zip(sample["ids"], sample["documents"], sample["metadatas"]), 1
):
    print(f"--- Record {i} ---")
    print(f"id:       {rid}")
    print(f"metadata: {meta}")
    print(f"text:     {doc[:200]}...\n")

--- Record 1 ---
id:       18f043028ff0ce20089b783283269729
metadata: {'label': 'Disney Q2 2026 Earnings', 'page_number': 1, 'chunk_type': 'text', 'chunk_index': 0, 'ingestion_timestamp': '2026-07-13T00:37:26.480085+00:00', 'section_title': 'UNITED STATES', 'source_file': 'form-10-q.pdf'}
text:     UNITED STATES

SECURITIES AND EXCHANGE COMMISSION Washington, D.C. 20549

FORM 10-Q

☒ QUARTERLY REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES EXCHANGE ACT OF 1934

For the quarterly period...

--- Record 2 ---
id:       6f742ac071834bc0a985c5b048783cba
metadata: {'chunk_type': 'text', 'source_file': 'form-10-q.pdf', 'page_number': 1, 'section_title': '500 South Buena Vista Street Burbank, California 91521', 'ingestion_timestamp': '2026-07-13T00:37:26.480085+00:00', 'label': 'Disney Q2 2026 Earnings', 'chunk_index': 1}
text:     urities registered pursuant to Section 12(b) of the Act:

Title of each class

Trading Symbol(s)

Name of each exchange on which registered

Common Stock,

## 4. Survey the metadata fields

See which metadata fields exist across the collection and the distinct values of the most useful
ones (label, chunk type, page range). This tells you what you can filter on.

In [14]:
from collections import Counter

all_meta = collection.get(include=["metadatas"])["metadatas"]

fields = set()
labels = Counter()
chunk_types = Counter()
pages = set()
for meta in all_meta:
    meta = meta or {}
    fields.update(meta.keys())
    labels[meta.get("label", "")] += 1
    chunk_types[meta.get("chunk_type", "")] += 1
    if "page_number" in meta:
        pages.add(meta["page_number"])

print("Metadata fields:", sorted(fields))
print("\nLabels:", dict(labels))
print("Chunk types:", dict(chunk_types))
if pages:
    print(f"Page range: {min(pages)}-{max(pages)}")

Metadata fields: ['chunk_index', 'chunk_type', 'ingestion_timestamp', 'label', 'page_number', 'section_title', 'source_file']

Labels: {'Disney Q2 2026 Earnings': 403}
Chunk types: {'text': 317, 'table': 86}
Page range: 1-87


## 5. Filter records by metadata

`collection.get(where=...)` restricts results to records matching a metadata filter. This is the
same mechanism the query pipeline uses to scope retrieval to a specific document.

In [15]:
# Example: only table chunks.
tables = collection.get(
    where={"chunk_type": "table"},
    include=["documents", "metadatas"],
    limit=3,
)
print(f"Showing {len(tables['ids'])} table chunk(s):\n")
for doc, meta in zip(tables["documents"], tables["metadatas"]):
    print(f"[page {meta.get('page_number')}] {doc[:150]}...\n")

# Other filters to try:
#   where={"source_file": "form-10-q.pdf"}
#   where={"label": "Disney Q2 2026 Earnings"}

Showing 3 table chunk(s):

[page 4] |                                                                             | Quarter Ended March 29,   | Quarter Ended March 29,   | Six Months End...

[page 5] |                                                                            | Quarter Ended   | Quarter Ended   | Six Months Ended   | Six Months End...

[page 6] |                                                                                                                            | March 28, 2026   | Sept...



## 6. Convenience: the `ChromaStore` helper

`rag.store.ChromaStore` wraps the **active** collection (`config.chromadb.collection_name`) and
exposes the same review helpers used by the notebooks and the Streamlit app — so you don't have to
aggregate metadata by hand.

In [16]:
store = ChromaStore(config)

print(f"Total chunks: {store.count()}\n")
print("Documents (file - label - chunks):")
dash = "—"
for d in store.list_documents():
    label = d.get("label") or dash
    print(f"  - {d['file']} - {label} - {d['chunks']} chunks")

# Per-document checks
src = "form-10-q.pdf"
print(f"\n'{src}' stored?    {store.document_exists(src)}")
print(f"'{src}' chunk count: {store.count_by_source(src)}")

Total chunks: 403

Documents (file - label - chunks):
  - form-10-q.pdf - Disney Q2 2026 Earnings - 403 chunks

'form-10-q.pdf' stored?    True
'form-10-q.pdf' chunk count: 403
